## Test01 of  SNNSNMetric

- author : Sylvie Dagoret-Campagne
- creation date : 2026-08-02
- kernel conda_py313_opsim53

In [ ]:
import os
import numpy as np
import pandas as pd
import healpy as hp

In [ ]:
from rubin_sim.maf.metrics import SNNSNMetric
from rubin_sim.maf.slicers import HealpixSlicer
from rubin_sim.maf.metric_bundles import MetricBundle
import rubin_sim.maf as maf

In [ ]:
# Voir les metrics existantes
# import rubin_sim.maf as maf
# dir(maf)

## Study SNNSNMetric()

### Define the Run

In [ ]:
path_topsim = os.getenv("RUBIN_SIM_DATA_DIR")
path_summary = os.path.join(path_topsim, "maf/fbs5.3/summary.h5")

In [ ]:
# opsdb = 'baseline_v5.3.5_10yrs.db'
opsdb = os.path.join(path_topsim, "baseline_v5.3.5_10yrs.db")
run_name = os.path.split(opsdb)[-1].replace(".db", "")

In [ ]:
print(f"inputfile {opsdb} for run {run_name}")

### Create a Metrics and the Slicer

In [ ]:
metric = SNNSNMetric()

slicer = HealpixSlicer(nside=64)
constraint = None
plot_dict = {"color_min": 0, "color_max": 1200, "extend": "max", "x_min": 0, "x_max": 1200}
plot_funcs = [maf.HealpixSkyMap(), maf.HealpixHistogram()]

# bundle = MetricBundle(metric, slicer, constraint='band="r"')
bundle = maf.MetricBundle(
    metric, slicer, constraint, run_name=run_name, plot_dict=plot_dict, plot_funcs=plot_funcs
)

👉 SNNSNMetric charge ses templates dès l’instanciation (ou lors de la préparation interne du bundle).

👉 Ce sont des light curves simulées de supernovae Ia
Paramètres encodés dans le nom :
-2.0, 0.0 → stretch / couleur (modèle SALT2)
380–800 nm → gamme spectrale
ebvofMW → extinction galactique

👉 Pour chaque fichier :
plusieurs filtres (g, r, i, z, y)
des grilles de redshift / phases

🔁 3. Pourquoi tous les filtres sont chargés alors que tu mets band="r" ?

👉 Point crucial :
Le constraint='band="r"' s’applique aux observations OpSim, PAS aux templates SN.

Donc :
✔️ tu filtres les observations (visites LSST en r)
❌ mais la métrique SN a besoin de toutes les bandes pour :
fitter les courbes de lumière
estimer redshift / distance
calculer la détection

👉 donc elle charge :
g, r, i, z, y

C’est normal et nécessaire physiquement.

In [ ]:
!ls /users/dagoret/DATA/OpSim/maf/SNe_data/

### 🚨 6. Point important (performance)

Ce que tu vois implique :

👉 SNNSNMetric est lourde

charge plusieurs gros fichiers HDF5
fait des interpolations multi-dimensionnelles
répété pour chaque pixel HEALPix

👉 donc :

CPU intensif
I/O important
mémoire non négligeable
🔬 7. Diagnostic rapide

Ton pipeline fait :

Initialisation métrique
Chargement templates SN
Préparation slicer

👉 MAIS tu n’as pas encore lancé :
```code
bundle.run()
```

### 1) Contexte : où se situe SNNSNMetric

Dans MAF (Metric Analysis Framework) de rubin_sim, une metric est une fonction :

Metric:{visites LSST} → quantite scientifique

MAF sert à évaluer une stratégie d’observation LSST .

👉 SNNSNMetric est une metric science cosmology qui répond à :

Combien de supernovae (Ia) LSST peut détecter et utiliser ?

### 2) Objectif physique de SNNSNMetric
#### 2.1 Ce que la metric calcule

Elle estime :

$N_{SN}(z)$ ou $N_{SN}$, detectees



en fonction de :

- cadence (sampling temporel)
- profondeur (m5)
- filtres
- conditions d’observation

👉 C’est donc une metric de détectabilité + qualité photométrique.

#### 2.2 Modèle astrophysique sous-jacent

Elle repose sur 4 briques physiques :

##### (A) Taux de SN Ia : $R_{SN}(z)\simeq SFR \otimes delay-time- distribution$

→ nombre de SN par volume et par temps

##### (B) Cosmologie → flux observé

$m=M+\mu(z)+K(z)+A_{dust}$

avec :

- $\mu(z)$ : distance modulus
- $K(z)$ : K-correction
- extinction MW (ebv)

##### (C) Courbes de lumière (light curves)

Les fichiers que tu vois :

`LC_-2.0_0.2_380.0_800.0_ebvofMW_0.0_vstack.hdf5`

contiennent :

- templates de SN Ia en fonction de :
- phase
- longueur d’onde
- stretch / color (type SALT2-like)

👉 Ce sont des grilles pré-calculées, pas générées à la volée.

##### (D) Détection → SNR

Pour chaque observation LSST :

$SNR=\frac{ F}{\sigma_{sky}}$


Puis :

- seuil de détection
- qualité de sampling (nombre de points avant/après pic)

### 3) Ce que fait réellement SNNSNMetric

Pipeline logique

Pour chaque pixel Healpix (ton slicer) :

##### Étape 1 — récupérer les visites
- 
                 dataSlice = observations in that pixel
  
##### Étape 2 — simuler des SN

Pour chaque redshift :

- tirer une SN
- choisir une light curve template
- appliquer cosmologie + dust
  
##### Étape 3 — projeter sur les visites

Pour chaque visite LSST :

- calculer flux attendu
- convertir en magnitude
- comparer à m5
  
##### Étape 4 — calcul SNR

$snr = \frac{flux}{flux\_error}$

##### Étape 5 — critères de sélection

Typiquement :

- ≥ N points détectés
- ≥ X points avant le pic
- SNR > seuil

##### Étape 6 — compter

if SN passes cuts:
    count += weight
    
Résultat final:

- nombre de SN détectables
- éventuellement distribution en redshift

### 4) Pourquoi tu vois ces fichiers HDF5

Quand tu fais :

metric = SNNSNMetric()

le constructeur :

👉 charge automatiquement :

- templates SN
- modèles d’erreur
- distributions

Depuis :

`/users/dagoret/DATA/OpSim/maf/SNe_data/`

Ces fichiers contiennent :
##### 1. LC_*.hdf5
- grilles de light curves
- multi-bandes (g,r,i,z,y)
- phases
##### 2. *_error_model_*.hdf5
- modèle de bruit
- incertitudes photométriques
##### 3. gamma_*.hdf5
- paramètres instrumentaux (bruit sky etc.)

👉 Ils sont :

- soit installés avec rubin_sim
- soit téléchargés automatiquement
- soit copiés via un setup précédent

### 5) Interprétation des logs que tu as

Exemple :

`Loading ... LC_-2.0_0.2_... g 20774 799 26`

Ça veut dire :

- filtre g
- 20774 points dans la grille
- 799 phases
- 26 longueurs d’onde (ou bins)

👉 Donc :

➡️ `SNNSNMetric` est data-driven

➡️ elle ne génère pas les SN → elle interpole des templates


### 6) Structure du code (schéma réaliste)

Dans rubin_sim.maf.metrics.snNSNMetric :

```python
class SNNSNMetric(BaseMetric):

    def __init__(...):
        self.load_templates()
        self.load_error_model()

    def run(self, dataSlice, slicePoint):

        sn_list = self.generate_sn_population()

        for sn in sn_list:
            lc = self.make_lightcurve(sn, dataSlice)
            snr = self.compute_snr(lc)

            if self.pass_cuts(lc, snr):
                count += weight

        return count

```



### 7) Points subtils (importants pour toi)

#### 7.1 Ce n’est PAS une simulation Monte Carlo complète

- pas de vraie génération continue
- interpolation de grilles pré-calculées

👉 compromis vitesse / réalisme

#### 7.2 Couplage fort avec OpSim

Les entrées critiques :

- fiveSigmaDepth
- observationStartMJD
- filter
- visitExposureTime
  
#### 7.3 Sensibilité extrême à la cadence

C’est LA physique clé :

👉 détecter une SN ≠ juste voir un point

il faut :

- pré-pic
- montée
- descente
  
### 8) Lecture physique finale

👉 `SNNSNMetric` mesure :

$ Efficiency = \frac{SN-utilisables}{SN-totales}$


- C’est donc une metric de science cosmologique, pas juste détection.


### 9) Résumé ultra-condensé

SNNSNMetric =

- prend un historique LSST
- injecte des SN Ia templates
- calcule leur observabilité
- applique des cuts réalistes
- retourne un nombre de SN exploitables

## Run and plot

In [ ]:
try_read = False
if try_read:
    bundle.read(bundle.file_root + ".npz")
    bundle.set_plot_dict(
        plot_dict={"color_min": 0, "color_max": 1200, "extend": "max", "x_min": 0, "x_max": 1200}
    )
    # bundle.set_plot_funcs(plot_funcs = [maf.HealpixSkyMap(),])

else:
    g = maf.MetricBundleGroup(
        {
            "SNNSNMetric": bundle,
        },
        opsdb,
        verbose=True,
    )
    g.run_all()

bundle.plot()

### 1. Vérifions le type du résultat

Après run_all() :

bundle.metric_values.dtype

Je m'attends à voir :

dtype('O')

ou :

object

C'est ce qui empêche le plot.

In [ ]:
bundle.metric_values.dtype

C'est ce qui empêche le plot.

### 2. Pourquoi SNNSNMetric donne un object ?

SNNSNMetric n'est pas une métrique de type "nombre de supernovae par pixel".

Elle retourne probablement un ensemble de valeurs ou une structure complexe.

Dans MAF, une métrique doit idéalement retourner :

un scalaire :
42.3

pour chaque pixel HEALPix

ou éventuellement :

un tableau numérique de taille fixe.

Mais SNNSNMetric est spéciale : elle calcule des informations liées aux light curves simulées de SN.

Elle peut retourner des objets du type :

dict
list
structured array

Donc :

metric_dtype="object"

a été défini dans la métrique.

### 3. Vérifier ce que contient un pixel

Fais :

```python
values = bundle.metric_values.compressed()
print(type(values[0]))
print(values[0])
```

Tu devrais voir quelque chose comme :

<class 'dict'>

ou :

<class 'numpy.ndarray'>


In [ ]:
values = bundle.metric_values.compressed()
print(type(values[0]))
print(values[0])

### 4. Quelle métrique utiliser pour une carte du nombre de SN ?

Si ton objectif est :

"Combien de SN Ia LSST détecte par pixel ?"

alors SNNSNMetric n'est probablement pas la bonne métrique directement.

Il faut une métrique qui retourne un nombre :

par exemple :

```python
metric = maf.metrics.SNNSNMetric()
```

calcule plutôt :

- nombre de saisons observées
- qualité des observations
- capacité à mesurer une SN
- cadence
- S/N

mais pas forcément un simple compteur.

### 5. Pour obtenir une carte HEALPix

**Il faut créer une métrique de réduction**.

Exemple :

```python
class SNCountMetric(BaseMetric):

    def __init__(self):
        super().__init__(
            col=["fieldRA"],
            metric_dtype=float
        )

    def run(self, data_slice, slice_point=None):

        return float(len(data_slice))
```

Alors :

metric_values.dtype

sera :

float64

et :

maf.HealpixSkyMap()

fonctionnera.

### 6. Concernant SNNSNMetric

Je pense que tu es tombée dans un piège classique de MAF.

Le nom :

SNNSNMetric

peut laisser penser :

"metric qui compte les supernovae"

mais en réalité elle appartient aux métriques SN Ia cosmology, et elle utilise les fichiers :

maf/SNe_data/
    LC_*.hdf5
    gamma_*.hdf5

que tu as vus précédemment.

Elle fait plutôt :

- prend un champ HEALPix
- injecte des SN simulées
- regarde quelles observations permettent une reconstruction
- calcule un indicateur SN

Ce n'est pas nécessairement un scalaire simple.

### 7. Pour confirmer

Peux-tu faire :

In [ ]:
# print(bundle.metric.metric_name)
print(bundle.metric.metric_dtype)

v = bundle.metric_values.compressed()

print(len(v))
print(type(v[0]))
print(v[0])

In [ ]:
dir(bundle.metric)